In [ ]:
#Run to determine PROJECT_DIR if we are on colab or locally
import os
if 'google.colab' in str(get_ipython()):
    # Running in Colab
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = "/content/drive/MyDrive/Teknisk fysik/Utbyte/Kurser/DAML/Project/notebooks"
else:
    # Running locally 
    PROJECT_DIR = '/Users/erikdalgard/Library/CloudStorage/GoogleDrive-dalgard.erik@gmail.com/My Drive/Teknisk fysik/Utbyte/Kurser/DAML/Project/notebooks'

os.chdir(PROJECT_DIR)


In [ ]:
#Importing useful packages
import pandas as pd
from tensorflow import keras
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger, BackupAndRestore
from datetime import datetime
import glob
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
from itertools import product

#Importing training_validation split function from cube_cutter
from cub_cutter_c import training_validation_split

#Importing froc evaluation functions from froc_eval
from froc_eval import evaluate, plot_froc, froc_table, compute_froc


#Importing modules
import models_2D, models_3D


In [ ]:
#Getting 2D CNN models
archi_1_2D = models_2D.get_archi_1_2D()
archi_2_2D = models_2D.get_archi_2_2D()
archi_3_2D = models_2D.get_archi_3_2D()

#Getting 3D CNN models
archi_1_3D = models_3D.get_archi_1_3D()
archi_2_3D = models_3D.get_archi_2_3D()
archi_3_3D = models_3D.get_archi_3_3D()

In [ ]:
#DEFINING THE TRAINING FUNCTION

def train_network(model, X_train, X_val=None, project_dir="", epochs=50, batch_size=64):
    """
    Compiles, logs, and trains a given Keras model.

    Parameters:
        model (keras.Model): The uncompiled Keras model to be trained.
        X_train: Training features.
        X_val: Validation features (Optional).
        project_dir (str): Directory for saving assets.
        epochs (int): Maximum number of training iterations.
        batch_size (int): Number of samples per training batch.

    Returns:
        History: The history as a keras object
        checkpoint_path: The file path to the weights of the model
        log_path: The file path to the csv logs of the training


    """

    loss_function = "binary_crossentropy"
    metrics_list = [
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="sensitivity"),
        keras.metrics.AUC(name="auc"),
    ]
    monitor_metric = "val_auc"
    monitor_mode = "max"

    # 2. Compile the model with the chosen settings and AdamW as optimizer
    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=1e-4,
            weight_decay=1e-4
        ),
        loss=loss_function,
        metrics=metrics_list
    )
    # 3. Create a clean subfolder dynamically named after the architecture
    model_folder = os.path.join(project_dir, 'training_history')
    history_folder = os.path.join(model_folder, model.name)
    os.makedirs(model_folder, exist_ok=True)
    os.makedirs(history_folder, exist_ok=True)

    time_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    checkpoint_path = os.path.join(history_folder, f'best_model_{time_stamp}.keras')
    log_path = os.path.join(history_folder, f'training_log_{time_stamp}.csv')

    # 4. Setting up automation callbacks
    callbacks = [
        ModelCheckpoint(
            filepath=checkpoint_path,
            monitor=monitor_metric,
            mode=monitor_mode,
            save_best_only=True,
            save_weights_only=False 
        ),
        EarlyStopping( #If model does not improve after 10 epochs, we stop training and restore best model
            monitor=monitor_metric,
            mode=monitor_mode,
            patience=5,
            restore_best_weights=True,
        ),
        CSVLogger(log_path), #logs the training

        ReduceLROnPlateau( #If we stop learning, we wait 5 epochs and then half the learning rate
            monitor=monitor_metric,
            mode=monitor_mode,
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1
        ),
        BackupAndRestore( #If google collabe crashes, this creates a backup folder with the weights of the previous training. If training is succeeded the backup file is deleted.
            backup_dir=os.path.join(history_folder, 'backup')
    ),
    ]


    # 5. Execute Training
    print(f"Launching training loop for: {model.name}")

    history = model.fit(
        X_train,
        validation_data=X_val,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1
    )


    print(f"\nSuccessfully finished training {model.name}!")
    print(f"Best weights secured at: {checkpoint_path}")
    print(f"History saved to: {log_path}")

    return history, checkpoint_path, log_path

In [ ]:
#FULL TRAINING LOOP

#Model list
model_list = [
    ("archi1", "2d", archi_1_2D),
    ("archi2", "2d", archi_2_2D),
    ("archi3", "2d", archi_3_2D),
    ("archi1", "3d", archi_1_3D),
    ("archi2", "3d", archi_2_3D),
    ("archi3", "3d", archi_3_3D),
]

for arch, layout, model in model_list:
    #for each model, get the training and validation split and train the network
    train_tfds, val_tfds = training_validation_split(PROJECT_DIR, arch, layout)
    train_network(model, X_train=train_tfds, X_val=val_tfds, project_dir=PROJECT_DIR)

In [ ]:
#LOADING ALL MODELS AFTER TRAINING
model_paths_2D = glob.glob("training_history/2D*/best_model_*.keras", recursive=True)
model_paths_3D = glob.glob("training_history/3D*/best_model_*.keras", recursive=True)

models_paths_combined = [("archi1", "2d", model_paths_2D[2]),
                         ("archi2", "2d", model_paths_2D[1]),
                         ("archi3", "2d", model_paths_2D[0]),
                         ("archi1", "3d", model_paths_3D[2]),
                         ("archi2", "3d", model_paths_3D[1]),
                         ("archi3", "3d", model_paths_3D[0]),
                         ]

#Loading the training paths
training_paths_2D = glob.glob("training_history/2D*/training_log*.csv", recursive=True)
training_paths_3D = glob.glob("training_history/3D*/training_log*.csv", recursive=True)

In [ ]:
# CALCULATING WEIGHTS BY GRID SEARCH AND GETTING FROC PLOT AND TABLE FROM 2D MODELS

subset_dir = Path(PROJECT_DIR) / "data/masked_scans3"

all_results = {}

for arch, layout, model_path in models_paths_combined[:3]:
    name = f"{arch}_{layout}"
    print(f"Evaluating {name}")
    model = keras.models.load_model(model_path)
    all_results[name] = evaluate(model, subset_dir, arch, layout)
    keras.backend.clear_session()

res1 = all_results["archi1_2d"]
res2 = all_results["archi2_2d"]
res3 = all_results["archi3_2d"]

# --- GRID SEARCH FOR BEST WEIGHTS ---
print("Running grid search for optimal ensemble weights...")
best_cpm, best_weights = 0, None

steps = np.arange(0, 1.1, 0.1)
for w1, w2, w3 in product(steps, repeat=3):
    if abs(w1 + w2 + w3 - 1.0) > 1e-6:
        continue
    y_ensemble = w1*res1["probs"] + w2*res2["probs"] + w3*res3["probs"]
    cpm = compute_froc(y_ensemble, res1["labels"], res1["n_scans"])["cpm"]
    if cpm > best_cpm:
        best_cpm = cpm
        best_weights = (round(w1,1), round(w2,1), round(w3,1))

w1, w2, w3 = best_weights
print(f"Best weights: γ1={w1}, γ2={w2}, γ3={w3}  (val CPM={best_cpm:.4f})")
# ------------------------------------

ensemble_probs = w1*res1["probs"] + w2*res2["probs"] + w3*res3["probs"]
res_ensemble = compute_froc(ensemble_probs, res1["labels"], res1["n_scans"])

fig, ax = plt.subplots(figsize=(7, 5))
plot_froc(res1, ax=ax, label=f"archi1      (CPM={res1['cpm']:.4f})")
plot_froc(res2, ax=ax, label=f"archi2      (CPM={res2['cpm']:.4f})")
plot_froc(res3, ax=ax, label=f"archi3      (CPM={res3['cpm']:.4f})")  # bug fix: was res2
plot_froc(res_ensemble, ax=ax, label=f"Ensemble  (CPM={res_ensemble['cpm']:.4f})")
ax.set_title("FROC comparison — 2D CNN models")
plt.tight_layout()
plt.show()

table = froc_table(
    ("Archi-1",  res1),
    ("Archi-2",  res2),
    ("Archi-3",  res3),
    ("Ensemble", res_ensemble),
)
print(table)

In [ ]:
# CALCULATING WEIGHTS BY GRID SEARCH AND GETTING FROC PLOT AND TABLE FROM 2D MODELS

subset_dir = Path(PROJECT_DIR) / "data/masked_scans3"

all_results = {}

for arch, layout, model_path in models_paths_combined[-3:]:
    name = f"{arch}_{layout}"
    print(f"Evaluating {name}")
    model = keras.models.load_model(model_path)
    all_results[name] = evaluate(model, subset_dir, arch, layout)
    keras.backend.clear_session()

res1 = all_results["archi1_3d"]
res2 = all_results["archi2_3d"]
res3 = all_results["archi3_3d"]

# --- GRID SEARCH FOR BEST WEIGHTS ---
print("Running grid search for optimal ensemble weights...")
best_cpm, best_weights = 0, None

steps = np.arange(0, 1.1, 0.1)
for w1, w2, w3 in product(steps, repeat=3):
    if abs(w1 + w2 + w3 - 1.0) > 1e-6:
        continue
    y_ensemble = w1*res1["probs"] + w2*res2["probs"] + w3*res3["probs"]
    cpm = compute_froc(y_ensemble, res1["labels"], res1["n_scans"])["cpm"]
    if cpm > best_cpm:
        best_cpm = cpm
        best_weights = (round(w1,1), round(w2,1), round(w3,1))

w1, w2, w3 = best_weights
print(f"Best weights: γ1={w1}, γ2={w2}, γ3={w3}  (val CPM={best_cpm:.4f})")
# ------------------------------------

ensemble_probs = w1*res1["probs"] + w2*res2["probs"] + w3*res3["probs"]
res_ensemble = compute_froc(ensemble_probs, res1["labels"], res1["n_scans"])

fig, ax = plt.subplots(figsize=(7, 5))
plot_froc(res1, ax=ax, label=f"archi1      (CPM={res1['cpm']:.4f})")
plot_froc(res2, ax=ax, label=f"archi2      (CPM={res2['cpm']:.4f})")
plot_froc(res3, ax=ax, label=f"archi3      (CPM={res3['cpm']:.4f})")  # bug fix: was res2
plot_froc(res_ensemble, ax=ax, label=f"Ensemble  (CPM={res_ensemble['cpm']:.4f})")
ax.set_title("FROC comparison — 3D CNN models")
plt.tight_layout()
plt.show()

table = froc_table(
    ("Archi-1",  res1),
    ("Archi-2",  res2),
    ("Archi-3",  res3),
    ("Ensemble", res_ensemble),
)
print(table)

In [ ]:
#Plotting auc and validation auc vs epochs for the 2D models
models = ['archi3', 'archi2', 'archi1']

for index, training_path in enumerate(training_paths_2D):
    df = pd.read_csv(training_path)
    plt.title(f"{models[index]}: Training and Validation AUC over Epochs")
    plt.plot(df['epoch'], df['auc'], label="AUC")
    plt.plot(df['epoch'], df['val_auc'], label="Validation AUC")
    plt.xlabel('Epochs')
    plt.ylabel("AUC")
    plt.legend()
    plt.grid()
    plt.savefig(f"{PROJECT_DIR}/plots/2D/{models[index]}")

    plt.close()

In [ ]:
#Plotting auc and validation auc vs epochs for the 3D models
models = ['archi3', 'archi2', 'archi1']
for index, training_path in enumerate(training_paths_3D):
    df = pd.read_csv(training_path)
    plt.title(f"{models[index]}: Training and Validation AUC over Epochs")
    plt.plot(df['epoch'], df['auc'], label="AUC")
    plt.plot(df['epoch'], df['val_auc'], label="Validation AUC")
    plt.xlabel('Epochs')
    plt.ylabel("AUC")
    plt.legend()
    plt.grid()
    plt.savefig(f"{PROJECT_DIR}/plots/3D/{models[index]}")

    plt.close()